# Teste de Processamento de linguagem natural

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import spacy
from nltk.corpus import stopwords
from transformers import pipeline

c:\Users\super\anaconda3\envs\tcc2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Download recursos
nltk.download('stopwords')
stop_words = set(stopwords.words('portuguese'))
nlp = spacy.load("pt_core_news_sm")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\super\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Pré-processamento do texto
# É preciso melhorar um pouco essa função, mas está rasoávelmente bem por enquanto
def limpar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r"http\S+|www\S+|https\S+", '', texto)
    texto = re.sub(r"\d+", '', texto)
    texto = re.sub(r"[^\w\s]", '', texto)
    tokens = [t for t in texto.split() if t not in stop_words]
    doc = nlp(" ".join(tokens))
    lemas = [token.lemma_ for token in doc]
    return " ".join(lemas)

In [6]:
# Carregar base e limpar texto
df = pd.read_csv("Data/olist_order_reviews_dataset.csv")
coluna_texto = "review_comment_message"
df = df.dropna(subset=[coluna_texto])
df['texto_limpo'] = df[coluna_texto].apply(limpar_texto)

In [11]:
df.head(30)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,texto_limpo
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,recebi bem antes prazo estipular
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,parabéns loja lannister adorar comprar Interne...
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22 00:00:00,2018-05-23 16:45:47,aparelho eficiente site marca aparelho impress...
12,4b49719c8a200003f700d3d986ea1a19,9d6f15f95d01e79bd1349cc208361f09,4,NaN,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",2018-02-16 00:00:00,2018-02-20 10:52:22,pouco travandopelo valor ta bom
15,3948b09f7c818e2d86c9a546758b2335,e51478e7e277a83743b6f9991dbfa3fb,5,Super recomendo,"Vendedor confiável, produto ok e entrega antes...",2018-05-23 00:00:00,2018-05-24 03:00:01,vendedor confiável produto ok entregar antes p...
16,9314d6f9799f5bfba510cc7bcd468c01,0dacf04c5ad59fd5a0cc1faa07c34e39,2,NaN,"GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E...",2018-01-18 00:00:00,2018-01-20 21:25:45,gostar saber sempre recebi compra agora decpci...
19,373cbeecea8286a2b66c97b1b157ec46,583174fbe37d3d5f0d6661be3aad1786,1,Não chegou meu produto,Péssimo,2018-08-15 00:00:00,2018-08-15 04:10:37,péssimo
22,d21bbc789670eab777d27372ab9094cc,4fc44d78867142c627497b60a7e0228a,5,Ótimo,Loja nota 10,2018-07-10 00:00:00,2018-07-11 14:10:25,lojar noto
24,0e0190b9db53b689b285d3f3916f8441,79832b7cb59ac6f887088ffd686e1d5e,5,NaN,obrigado pela atençao amim dispensada,2017-12-01 00:00:00,2017-12-09 22:58:58,obrigar atençao amim dispensar
27,fe3db7c069d694bab50cc43463f91608,2ca73e2ff9e3a186ad1e1ffb9b1d9c10,5,NaN,A compra foi realizada facilmente.\r\nA entreg...,2018-03-23 00:00:00,2018-04-01 00:27:51,compra realizar facilmente entregar efetuar an...


In [16]:
# Pipeline de classificação de emoções/sentimentos
classifier = pipeline(
    "text-classification",
    model="pltoledo/my_awesome_model",
    tokenizer="pltoledo/my_awesome_model",
    top_k=None,
    device=-1
)

Device set to use cpu


In [13]:
# Classifica e retorna emoções com scores
def classificar_emocoes(texto):
    resultados = classifier(texto)[0]
    filtradas = [(r['label'], r['score']) for r in resultados if r['score'] >= 0.5]
    return filtradas if filtradas else [(resultados[0]['label'], resultados[0]['score'])]

df['emocao_scores'] = df['texto_limpo'].apply(classificar_emocoes)

KeyboardInterrupt: 

In [13]:
# Mapeamento das emoções para uma escala 1-5 (exemplo)
mapeamento = {
    'joy': 5,
    'love': 5,
    'surprise': 4,
    'neutral': 3,
    'sadness': 2,
    'anger': 1,
    'fear': 1,
    # Adapte conforme as emoções do modelo
}

In [14]:
def mapear_satisfacao(lista_emocoes):
    # Se mais de uma emoção, calcula média ponderada
    soma_peso = 0
    soma_score = 0
    for label, score in lista_emocoes:
        peso = mapeamento.get(label.lower(), 3)  # neutro padrão
        soma_peso += peso * score
        soma_score += score
    return round(soma_peso / soma_score, 2) if soma_score > 0 else 3

In [15]:
df['nivel_satisfacao'] = df['emocao_scores'].apply(mapear_satisfacao)

In [20]:
df.head(100)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,texto_limpo,emocao_scores,nivel_satisfacao
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,recebi bem antes prazo estipular,"[(neutral, 0.7800531387329102)]",3.0
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,parabéns loja lannister adorar comprar Interne...,"[(love, 0.6320717334747314)]",5.0
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22 00:00:00,2018-05-23 16:45:47,aparelho eficiente site marca aparelho impress...,"[(neutral, 0.8150433301925659)]",3.0
12,4b49719c8a200003f700d3d986ea1a19,9d6f15f95d01e79bd1349cc208361f09,4,NaN,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",2018-02-16 00:00:00,2018-02-20 10:52:22,pouco travandopelo valor ta bom,"[(neutral, 0.3537318706512451)]",3.0
15,3948b09f7c818e2d86c9a546758b2335,e51478e7e277a83743b6f9991dbfa3fb,5,Super recomendo,"Vendedor confiável, produto ok e entrega antes...",2018-05-23 00:00:00,2018-05-24 03:00:01,vendedor confiável produto ok entregar antes p...,"[(neutral, 0.4052439332008362)]",3.0
...,...,...,...,...,...,...,...,...,...,...
236,c4a20a1d2d634f814cca9311edd60e99,1a558554b7f10eba1ffdd56ce469df63,5,NaN,"otimo produto só nao veio manual, mas sem prob...",2018-04-21 00:00:00,2018-04-24 14:26:37,otimo produto nao vir manual problema,"[(admiration, 0.749000072479248)]",3.0
237,81e7291e2f92b492eb0340f4406c6897,93720f5a467d2edd5e84e209e8a9a4f6,4,NaN,"Uma das peças não encaixava, estava de diâmetr...",2018-04-19 00:00:00,2018-04-22 16:36:04,peça encaixar diâmetro diferente serrar martel...,"[(neutral, 0.8190307021141052)]",3.0
238,eb6d75bedecae7b3dbea11b8530637d8,00b9e0f8f588d0406a3de447eb606970,5,NaN,Excelente produto.,2018-02-17 00:00:00,2018-02-19 16:20:08,excelente produto,"[(admiration, 0.8036838173866272)]",3.0
239,593f9b33c3a4c74e37b7b7c49f0cedc4,268762d6a983092f8632e379836806eb,5,NaN,Tudo certo,2017-09-07 00:00:00,2017-09-13 13:51:51,tudo certo,"[(neutral, 0.33277231454849243)]",3.0


In [17]:
# Exportar uma amostra para validação manual
df.sample(100)[[coluna_texto, 'texto_limpo', 'emocao_scores', 'nivel_satisfacao']].to_csv(
    "amostra_validacao_satisfacao.csv", index=False, encoding='utf-8-sig')

In [18]:
print("Amostra para validação gerada: 'amostra_validacao_satisfacao.csv'")
print(df[['nivel_satisfacao']].describe())

Amostra para validação gerada: 'amostra_validacao_satisfacao.csv'
       nivel_satisfacao
count      41753.000000
mean           3.081482
std            0.418777
min            1.000000
25%            3.000000
50%            3.000000
75%            3.000000
max            5.000000


In [19]:
df['nivel_satisfacao'].value_counts()

nivel_satisfacao
3.00    39534
5.00     1769
2.00      257
4.00       37
1.00       24
4.02        9
4.13        8
4.01        7
4.06        7
3.89        6
4.04        6
3.91        6
4.05        6
4.07        5
4.09        5
4.08        4
3.95        4
3.84        4
3.96        4
3.97        4
4.14        4
3.93        4
3.99        3
3.85        3
3.90        3
4.11        3
4.12        3
4.10        3
3.92        3
4.03        3
3.88        3
3.87        2
4.17        2
3.98        2
3.94        2
3.66        1
3.83        1
3.86        1
3.82        1
Name: count, dtype: int64

pip uninstall torch torchvision torchaudio -y
pip install torch-directml transformers

import torch_directml
from transformers import pipeline

# Cria o device do DirectML
dml_device = torch_directml.device()

# Carrega o pipeline usando DirectML
classifier = pipeline(
    task="text-classification",
    model="pltoledo/my_awesome_model",
    tokenizer="pltoledo/my_awesome_model",
    top_k=None,
    device=dml_device
)